# NAIC WP7 UC5: Graph-Based Classification of AIS Time-Series Data

**Graph Neural Networks for maritime vessel activity classification**

> **Key Result:** GraphSAGE achieved **94.4% test accuracy** on fishing vs. non-fishing classification,
> outperforming both GCN and GAT architectures on AIS time-series data.

This demonstrator walks through the full pipeline:
1. Understanding AIS time-series data
2. Converting time-series to graph structures
3. Training three GNN architectures (GCN, GraphSAGE, GAT)
4. Comparing model performance
5. Visualizing learned representations

> **Note:** This notebook uses synthetic data for the data exploration and training sections to be self-contained. The accuracy bar chart in the summary uses real results from the full AIS dataset (23,500 samples).

**Authors:** Xue-Cheng Tai & Gro Fonnes (NORCE)

## 0. Setup

If running on an NAIC Orchestrator VM, the environment should already be configured via `setup.sh`.
Otherwise, install dependencies with `pip install -r requirements.txt && pip install -e .`

In [ ]:
import os
os.environ['DGLBACKEND'] = 'pytorch'

import numpy as np
import torch
import dgl
import matplotlib.pyplot as plt
import matplotlib
from dgl.dataloading import GraphDataLoader

import sys
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from graph_classification.models import GCN, GAT, GraphSAGE
from graph_classification.heads import GraphClassificationHead
from graph_classification.ais_timeseries_dataset import AISTimeseriesDataset
from graph_classification.utils import (
    create_region_force_model, process_one, get_test_result,
    create_ais_classification_model_df, plot_metrics
)

print(f'PyTorch {torch.__version__}')
print(f'DGL {dgl.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 1. Understanding AIS Time-Series Data (Illustrative)

The **Automatic Identification System (AIS)** broadcasts vessel information including
position, speed, heading, and course. From raw AIS data, three features are extracted
for each of 12 time steps:

| Feature | Description | Why it matters |
|---------|-------------|----------------|
| **Velocity** | Speed of the vessel | Fishing vessels have lower, more variable speeds |
| **Distance to shore** | Proximity to coastline | Fishing occurs in specific zones |
| **Curvature** | Rate of course change | Fishing involves frequent turns |

Each sample is a tensor of shape `(3, 12)` — 3 features over 12 time steps.

> **Note:** The patterns below are generated to illustrate the general characteristics of fishing vs non-fishing trajectories. Real AIS data shows similar but noisier distributions.

Let's generate synthetic data that mimics these patterns.

In [ ]:
def generate_synthetic_ais_data(n_samples=2000, seed=42):
    """Generate synthetic AIS data mimicking fishing vs non-fishing patterns.
    
    Fishing vessels: lower speed, closer to shore, higher curvature (more turns)
    Non-fishing vessels: higher speed, farther from shore, lower curvature (straight lines)
    """
    rng = np.random.RandomState(seed)
    n_timesteps = 12
    n_features = 3  # velocity, distance_to_shore, curvature
    
    X = np.zeros((n_samples, n_features, n_timesteps), dtype=np.float32)
    y = np.zeros(n_samples, dtype=np.float32)
    
    n_fishing = n_samples // 2
    
    # Fishing vessels (label = 1)
    for i in range(n_fishing):
        speed = rng.uniform(2, 6, n_timesteps) + rng.normal(0, 0.5, n_timesteps)  # low, variable speed
        shore_dist = rng.uniform(1, 15, n_timesteps) + rng.normal(0, 1, n_timesteps)  # close to shore
        curvature = rng.uniform(0.3, 1.0, n_timesteps) + rng.normal(0, 0.1, n_timesteps)  # high curvature
        X[i] = np.array([speed, shore_dist, curvature])
        y[i] = 1.0
    
    # Non-fishing vessels (label = 0)
    for i in range(n_fishing, n_samples):
        speed = rng.uniform(8, 20, n_timesteps) + rng.normal(0, 0.3, n_timesteps)  # high, steady speed
        shore_dist = rng.uniform(10, 50, n_timesteps) + rng.normal(0, 2, n_timesteps)  # far from shore
        curvature = rng.uniform(0.0, 0.3, n_timesteps) + rng.normal(0, 0.05, n_timesteps)  # low curvature
        X[i] = np.array([speed, shore_dist, curvature])
        y[i] = 0.0
    
    # Shuffle
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

X_all, y_all = generate_synthetic_ais_data(2000)
print(f'Dataset shape: X={X_all.shape}, y={y_all.shape}')
print(f'Class distribution: fishing={int(y_all.sum())}, non-fishing={int((1-y_all).sum())}')

### Visualize sample patterns (illustrative)

Let's look at what fishing vs. non-fishing time-series look like across the three features.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
feature_names = ['Velocity', 'Distance to Shore', 'Curvature']
colors = {'Fishing': '#e74c3c', 'Non-fishing': '#3498db'}

# Pick 5 samples of each class
fishing_idx = np.where(y_all == 1)[0][:5]
nonfishing_idx = np.where(y_all == 0)[0][:5]

for feat_i, (ax, name) in enumerate(zip(axes, feature_names)):
    for idx in fishing_idx:
        ax.plot(range(12), X_all[idx, feat_i, :], color=colors['Fishing'], alpha=0.5, linewidth=1.5)
    for idx in nonfishing_idx:
        ax.plot(range(12), X_all[idx, feat_i, :], color=colors['Non-fishing'], alpha=0.5, linewidth=1.5)
    ax.set_xlabel('Time Step')
    ax.set_ylabel(name)
    ax.set_title(name)
    ax.grid(True, alpha=0.3)

# Legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color=colors['Fishing'], label='Fishing'),
                   Line2D([0], [0], color=colors['Non-fishing'], label='Non-fishing')]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, fontsize=12, bbox_to_anchor=(0.5, 1.05))
plt.tight_layout()
plt.show()

print('Fishing: low speed, close to shore, high curvature (frequent turns)')
print('Non-fishing: high speed, far from shore, low curvature (straight transit)')

## 2. From Time-Series to Graphs

Each time-series is converted to a **DGL graph** where:
- **12 nodes** represent the 12 time steps
- **11 edges** connect consecutive time steps (chain structure)
- **12 self-loops** ensure every node receives its own features
- **Node features** are the 3-dimensional vectors (velocity, shore distance, curvature)

This converts the sequential data into a graph structure that GNNs can process.

In [ ]:
# Split into train/val/test
n_train = 1200
n_val = 400
n_test = 400

# Create datasets using temp files (required by DGLDataset)
import tempfile

def make_dgl_dataset(X, y, name, tmpdir):
    x_path = os.path.join(tmpdir, f'{name}_X.npy')
    y_path = os.path.join(tmpdir, f'{name}_y.npy')
    np.save(x_path, X)
    np.save(y_path, y)
    return AISTimeseriesDataset(name=name, raw_x_file=x_path, raw_y_file=y_path, save_dir=tmpdir)

tmpdir = tempfile.mkdtemp()
train_ds = make_dgl_dataset(X_all[:n_train], y_all[:n_train], 'train', tmpdir)
val_ds = make_dgl_dataset(X_all[n_train:n_train+n_val], y_all[n_train:n_train+n_val], 'val', tmpdir)
test_ds = make_dgl_dataset(X_all[n_train+n_val:], y_all[n_train+n_val:], 'test', tmpdir)

print(f'Training:   {len(train_ds)} graphs')
print(f'Validation: {len(val_ds)} graphs')
print(f'Test:       {len(test_ds)} graphs')

# Inspect one graph
g, label = train_ds[0]
print(f'\nSample graph:')
print(f'  Nodes: {g.num_nodes()}')
print(f'  Edges: {g.num_edges()} (11 sequential + 12 self-loops)')
print(f'  Node features shape: {g.ndata["attr"].shape}')
print(f'  Label: {"fishing" if label == 1 else "non-fishing"}')

### Visualize the graph structure

Let's see what one of these time-series graphs looks like.

In [ ]:
import networkx as nx

g_sample, label_sample = train_ds[0]

# Remove self-loops for cleaner visualization
g_vis = dgl.remove_self_loop(g_sample)
nx_g = g_vis.to_networkx().to_undirected()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Graph structure
ax = axes[0]
pos = {i: (i, 0) for i in range(12)}  # Linear layout
node_colors = g_sample.ndata['attr'][:, 0].numpy()  # Color by velocity
nx.draw(nx_g, pos, ax=ax, with_labels=True, node_color=node_colors,
        cmap=plt.cm.RdYlBu_r, node_size=500, font_size=9, font_weight='bold',
        edge_color='gray', width=2)
ax.set_title(f'Graph Structure (colored by velocity)\nLabel: {"Fishing" if label_sample == 1 else "Non-fishing"}',
             fontsize=12)

# Right: Node features as heatmap
ax = axes[1]
features = g_sample.ndata['attr'].numpy().T  # (3, 12)
im = ax.imshow(features, aspect='auto', cmap='viridis')
ax.set_xlabel('Time Step (Node)')
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['Velocity', 'Shore Dist', 'Curvature'])
ax.set_title('Node Features Heatmap', fontsize=12)
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## 3. GNN Architectures

We compare three Graph Neural Network architectures:

| Model | Mechanism | Parameters | Strengths |
|-------|-----------|:----------:|:---------:|
| **GCN** | Spectral convolutions | 8,832 | Simple, effective on fixed graphs |
| **GraphSAGE** | Mean neighborhood sampling | 17,216 | Scalable, inductive learning |
| **GAT** | Multi-head attention | 13,248 | Learns neighbor importance |

All three follow the same pipeline:
1. **GNN backbone** → node embeddings via message passing
2. **Graph readout** → mean pooling over nodes
3. **Classification head** → binary prediction (fishing / non-fishing)

In [ ]:
# Show model architectures and parameter counts
dim_nfeats = train_ds.dim_nfeats
gclasses = train_ds.gclasses
print(f'Input features: {dim_nfeats}, Output classes: {gclasses}\n')

for model_name in ['GCN', 'GSG', 'GAT']:
    rf_model, hidden = create_region_force_model('cpu', dim_nfeats, model_name)
    n_params = sum(p.numel() for p in rf_model.parameters() if p.requires_grad)
    print(f'{model_name:>3s}: {type(rf_model).__name__:<12s} | hidden={hidden:>3d} | params={n_params:,}')

## 4. Training All Models

We train each architecture with two learning rates and use early stopping
based on validation accuracy.

In [ ]:
from graph_classification.train_graph_classification_ais import train

models_to_train = ['GCN', 'GSG', 'GAT']
learning_rates = [0.01, 0.025]
epochs = 60
patience = 15
batch_size = 600

model_res = {}

for model_name in models_to_train:
    for lr in learning_rates:
        print(f'\n--- Training {model_name} (lr={lr}) ---')
        best_acc, losses, val_accs, test_accs = train(
            device=device,
            train_ds=train_ds,
            val_ds=val_ds,
            test_ds=test_ds,
            seed=0,
            model=model_name,
            lr=lr,
            epochs=epochs,
            patience=patience,
            batch_size=batch_size,
            model_path=None,
            pin_memory=False,
            num_workers=0,
        )
        model_res[f'{model_name}_lr_{lr}'] = {
            'model': model_name,
            'learning_rate': lr,
            'losses': losses,
            'val_accs': val_accs,
            'test_accs': test_accs,
            'best_test_acc': best_acc,
        }
        print(f'  Best test accuracy: {best_acc:.4f}')

## 5. Results Comparison

### Accuracy Table

In [ ]:
result_df = create_ais_classification_model_df(
    model_res, model_key='model', lr_key='learning_rate', acc_key='best_test_acc'
)
print('Test Accuracy by Model and Learning Rate:')
print('=' * 50)
print(result_df.to_string())
print()

# Find best
best_key = max(model_res, key=lambda k: model_res[k]['best_test_acc'])
best = model_res[best_key]
print(f'Best model: {best["model"]} (lr={best["learning_rate"]}) with {best["best_test_acc"]:.4f} accuracy')

### Training Curves

Loss, validation accuracy, and test accuracy over training epochs for each model.

In [ ]:
fig, axes = plt.subplots(len(models_to_train), 3, figsize=(16, 4 * len(models_to_train)))

for row, model_name in enumerate(models_to_train):
    for lr_idx, lr in enumerate(learning_rates):
        key = f'{model_name}_lr_{lr}'
        data = model_res[key]
        color = f'C{lr_idx}'
        label = f'lr={lr}'
        
        axes[row, 0].plot(data['losses'], color=color, label=label, alpha=0.8)
        axes[row, 1].plot(data['val_accs'], color=color, label=label, alpha=0.8)
        axes[row, 2].plot(data['test_accs'], color=color, label=label, alpha=0.8)
    
    axes[row, 0].set_title(f'{model_name} — Loss')
    axes[row, 0].set_xlabel('Epoch')
    axes[row, 0].set_ylabel('Loss')
    axes[row, 0].legend()
    axes[row, 0].grid(True, alpha=0.3)
    
    axes[row, 1].set_title(f'{model_name} — Validation Accuracy')
    axes[row, 1].set_xlabel('Epoch')
    axes[row, 1].set_ylabel('Accuracy')
    axes[row, 1].legend()
    axes[row, 1].grid(True, alpha=0.3)
    
    axes[row, 2].set_title(f'{model_name} — Test Accuracy')
    axes[row, 2].set_xlabel('Epoch')
    axes[row, 2].set_ylabel('Accuracy')
    axes[row, 2].legend()
    axes[row, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Real AIS Dataset Results

The following results are from the original DGL_Demonstrator notebook trained on the full AIS dataset (23,500 samples: 14,100 train, 4,700 val, 4,700 test).

In [ ]:
# Real results from full AIS dataset (notebooks/DGL_Demonstrator.ipynb)
print("=" * 65)
print("Results on Real AIS Dataset (23,500 samples)")
print("=" * 65)
print(f"{'Model':<12} {'LR':>6} {'Val Acc':>10} {'Test Acc':>10} {'Epochs':>8}")
print("-" * 65)
real_results = [
    ("GCN",      0.010, 0.93553, 0.94426, 37),
    ("GCN",      0.025, 0.93468, 0.94234, 29),
    ("GSG",      0.010, 0.93596, 0.94447, 52),
    ("GSG",      0.025, 0.93596, 0.94447, 53),
    ("GAT",      0.010, 0.92191, 0.93149, 55),
    ("GAT",      0.025, 0.85681, 0.86340, 44),
]
for model, lr, val_acc, test_acc, epochs in real_results:
    marker = " ★" if test_acc > 0.944 else ""
    print(f"{model:<12} {lr:>6.3f} {val_acc:>10.4f} {test_acc:>10.4f} {epochs:>8d}{marker}")
print("-" * 65)
print("★ = Best performing configurations")
print("\nBest Model: GraphSAGE (GSG) with lr=0.01 — 94.4% test accuracy")

### Model Comparison Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

model_names = []
accuracies = []
bar_colors = []
color_map = {'GCN': '#2ecc71', 'GSG': '#3498db', 'GAT': '#e74c3c'}

for key, data in sorted(model_res.items()):
    model_names.append(f'{data["model"]}\nlr={data["learning_rate"]}')
    accuracies.append(data['best_test_acc'] * 100)
    bar_colors.append(color_map[data['model']])

bars = ax.bar(model_names, accuracies, color=bar_colors, edgecolor='white', linewidth=1.5)

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('GNN Model Comparison on AIS Classification', fontsize=14)
ax.set_ylim(bottom=max(0, min(accuracies) - 10), top=100)
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
ax.legend()

plt.tight_layout()
plt.show()

## 6. Understanding the Graph Neural Network Pipeline

The classification pipeline has three stages. Let's trace a single graph through the network.

In [ ]:
# Trace a single graph through the GCN pipeline
g_trace, label_trace = test_ds[0]
g_trace = g_trace.to(device)

# Build the full pipeline
init_conv = GCN(dim_nfeats, gclasses, depth=2).to(device)
rf_model, hidden = create_region_force_model(device, dim_nfeats, 'GCN')
head0 = GraphClassificationHead(gclasses, gclasses).to(device)
head = GraphClassificationHead(hidden, gclasses).to(device)

with torch.no_grad():
    # Stage 1: Initial convolution
    h_init = init_conv(g_trace, g_trace.ndata['attr'].float())
    print(f'Stage 1 — Initial GCN convolution:')
    print(f'  Input:  {g_trace.ndata["attr"].shape} (12 nodes x 3 features)')
    print(f'  Output: {h_init.shape} (12 nodes x {h_init.shape[1]} features)')
    
    # Stage 2: Region force model (deeper GCN)
    h_rf = rf_model(g_trace, g_trace.ndata['attr'].float())
    print(f'\nStage 2 — Region force GCN (depth=3):')
    print(f'  Output: {h_rf.shape} (12 nodes x {h_rf.shape[1]} features)')
    
    # Stage 3: Graph readout + classification
    pred0 = head0(g_trace, h_init)
    pred = head(g_trace, h_rf)
    print(f'\nStage 3 — Graph readout (mean pooling) + classification:')
    print(f'  Graph-level prediction: {pred.shape} → classes: {pred.cpu().numpy().flatten()}')
    print(f'  Predicted class: {pred.argmax().item()} ({"fishing" if pred.argmax().item() == 1 else "non-fishing"})')
    print(f'  True label: {int(label_trace)} ({"fishing" if label_trace == 1 else "non-fishing"})')

## 7. Summary

The accuracy figures in this table are from the **real AIS dataset** (23,500 samples), not the synthetic data used in earlier sections.

| Property | GCN | GraphSAGE (GSG) | GAT |
|----------|:---:|:---------------:|:---:|
| **Mechanism** | Spectral convolution | Mean aggregation | Attention-weighted |
| **Parameters** | 8,832 | 17,216 | 13,248 |
| **Training speed** | Fast | Fast | Slower (attention) |
| **Sensitivity to LR** | Low | Low | High |
| **Best accuracy** | 94.4% | 94.4% | 93.1% |

### Key Takeaways

1. **Graph structure captures temporal patterns** — converting AIS time-series to graphs
   enables GNNs to learn vessel behavior patterns effectively.

2. **GCN and GraphSAGE perform comparably** — both achieve 94.4% on the real test set (4,700 samples),
   with GraphSAGE being more robust across learning rates.

3. **GAT is more sensitive to hyperparameters** — the attention mechanism adds
   complexity but doesn't improve accuracy on this relatively simple graph structure.

4. **Early stopping is effective** — models converge within 30-55 epochs, and
   patience-based stopping prevents overfitting.

### Using Real AIS Data

To use the actual AIS dataset (23,500 samples), place the `.npy` files in `data/`
and use the original notebook at `notebooks/DGL_Demonstrator.ipynb`, or run from CLI:

```bash
export PYTHONPATH=$PYTHONPATH:$(pwd)/src
python src/graph_classification/train_graph_classification_ais.py --data_folder data/ --epochs 100
```

---

*NAIC Use Case 5 — Developed by NORCE Norwegian Research Centre*